# RAG: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

Document: Gloomhaven rulebook. Both pipelines share the **same chunks and the same top-10 candidates**; only the ordering of the top 3 differs.

1. Setup
2. Load PDF and chunk
3. Embed and index
4. Pipeline 1 (baseline) and Pipeline 2 (re-rank)
5. Failure-mode queries
6. Latency profiling
7. Export tables for the LaTeX report

## 1. Setup

In [1]:
import os, re, time, json, getpass
import numpy as np
import pandas as pd
import pymupdf
from rapidocr_onnxruntime import RapidOCR
from sentence_transformers import SentenceTransformer, CrossEncoder
from google import genai
from google.genai import types

PDF_PATH = "data/Gloomhaven-2025-Rulebook.pdf"
BI_ENCODER = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GEMINI_MODEL = "gemini-3.5-flash-lite"  # flash models hit free-tier quota (429) or timed out on this key
CHUNK_WORDS, OVERLAP_WORDS = 200, 40
TOP_K, TOP_N = 10, 3  # retrieve 10, send 3 to the LLM

C:\Users\Lenovo\Desktop\RAG Assignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# The key is read from the environment or typed in; it is never stored in the notebook.
api_key = os.environ.get("GEMINI_API_KEY") or getpass.getpass("Gemini API key: ")
client = genai.Client(api_key=api_key, http_options=types.HttpOptions(timeout=60000))

bi_encoder = SentenceTransformer(BI_ENCODER)
cross_encoder = CrossEncoder(CROSS_ENCODER, max_length=512)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1216.74it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2102.34it/s]

## 2. Load PDF and chunk

Fixed-size word windows with overlap, so a rule that straddles a boundary still appears whole in at least one chunk. Each chunk keeps its starting page for citation.

In [3]:
# The PDF is a scan with no text layer, so each page is OCR'd once and cached.
OCR_CACHE = "data/ocr_pages.json"


def ocr_page(page, engine):
    pix = page.get_pixmap(dpi=150)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)[:, :, :3]
    found, _ = engine(img)
    boxes = [(b[0][0], b[0][1], t) for b, t, conf in (found or []) if float(conf) > 0.6]
    mid = pix.width / 2  # two-column layout: read the left column first, then the right
    left = sorted((b for b in boxes if b[0] < mid), key=lambda b: b[1])
    right = sorted((b for b in boxes if b[0] >= mid), key=lambda b: b[1])
    text = " ".join(t for _, _, t in left + right)
    return re.sub(r"\s+", " ", text).strip()


if os.path.exists(OCR_CACHE):
    with open(OCR_CACHE, encoding="utf-8") as f:
        pages = [tuple(p) for p in json.load(f)]
else:
    engine = RapidOCR()
    pages = []
    for page_no, page in enumerate(pymupdf.open(PDF_PATH), start=1):
        pages.append((page_no, ocr_page(page, engine)))
        print(f"OCR page {page_no}", end="\r")
    with open(OCR_CACHE, "w", encoding="utf-8") as f:
        json.dump(pages, f)

print(f"{len(pages)} pages, {sum(len(t.split()) for _, t in pages)} words")

74 pages, 29247 words


In [4]:
words, word_page = [], []
for page_no, text in pages:
    w = text.split()
    words += w
    word_page += [page_no] * len(w)

step = CHUNK_WORDS - OVERLAP_WORDS
chunks = []
for start in range(0, len(words), step):
    end = min(start + CHUNK_WORDS, len(words))
    chunks.append({"id": f"C{len(chunks):03d}", "page": word_page[start], "text": " ".join(words[start:end])})
    if end == len(words):
        break

print(len(chunks), "chunks")
print(chunks[5]["id"], "page", chunks[5]["page"], "->", chunks[5]["text"][:300])

183 chunks
C005 page 4 -> a thematic description of your character and a section number, which is only read once the character retires (see p. 55). Character Standee: This standee is an alternative to your plastic miniature. You can use either the standee or the miniature to represent your character, based on your preference


## 3. Embed and index

Vectors are L2-normalised, so a plain dot product equals cosine similarity. The "vector store" is a NumPy matrix.

In [5]:
doc_emb = bi_encoder.encode(
    [c["text"] for c in chunks], normalize_embeddings=True, batch_size=32, show_progress_bar=True
)
print(doc_emb.shape)

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Batches:  17%|█▋        | 1/6 [00:01<00:09,  1.82s/it]

Batches:  33%|███▎      | 2/6 [00:03<00:06,  1.70s/it]

Batches:  50%|█████     | 3/6 [00:04<00:04,  1.56s/it]

Batches:  67%|██████▋   | 4/6 [00:06<00:02,  1.47s/it]

Batches:  83%|████████▎ | 5/6 [00:07<00:01,  1.43s/it]

Batches: 100%|██████████| 6/6 [00:08<00:00,  1.27s/it]

Batches: 100%|██████████| 6/6 [00:08<00:00,  1.42s/it]

(183, 384)


## 4. The two pipelines

Both use the identical prompt via `ask_gemini`, so any difference in the answer comes from which 3 chunks were supplied.

In [6]:
def baseline_retrieve(query, k=TOP_K):
    """Pipeline 1 stage 1: top-k chunk indices by cosine similarity."""
    q = bi_encoder.encode(query, normalize_embeddings=True)
    scores = doc_emb @ q
    idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in idx]


def rerank(query, candidates):
    """Pipeline 2: score every (query, chunk) pair jointly and re-sort descending."""
    pairs = [(query, chunks[i]["text"]) for i, _ in candidates]
    ce_scores = cross_encoder.predict(pairs)
    order = np.argsort(-ce_scores)
    return [(candidates[j][0], float(ce_scores[j])) for j in order]


PROMPT = (
    "Answer the question using ONLY the context below. If the context does not contain the answer, "
    "say so. Cite chunk IDs in brackets.\n\n"
    "Context:\n{context}\n\nQuestion: {query}\nAnswer:"
)


def ask_gemini(query, chunk_idx, retries=5):
    context = "\n\n".join(f"[{chunks[i]['id']}] (p.{chunks[i]['page']}) {chunks[i]['text']}" for i in chunk_idx)
    cfg = types.GenerateContentConfig(temperature=0)
    for attempt in range(retries):
        try:
            r = client.models.generate_content(
                model=GEMINI_MODEL, contents=PROMPT.format(context=context, query=query), config=cfg
            )
            return r.text
        except Exception as e:  # free tier rate limits surface as generic API errors
            wait = 15 * (attempt + 1)
            print(f"Gemini error ({e}); retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError("Gemini call failed after retries")

In [7]:
def gold_ids(phrase):
    """Chunks that contain the answer phrase (found by reading the rulebook)."""
    p = phrase.lower()
    return {i for i, c in enumerate(chunks) if p in c["text"].lower()}


def rank_of(gold, ranking):
    """1-based rank of the best-placed gold chunk, or None if absent from the list."""
    for pos, (i, _) in enumerate(ranking, start=1):
        if i in gold:
            return pos
    return None


def run_query(query, gold_phrase=None, answer=True):
    candidates = baseline_retrieve(query)
    reranked = rerank(query, candidates)
    out = {"query": query, "baseline": candidates, "reranked": reranked}
    if gold_phrase:
        g = gold_ids(gold_phrase)
        out["gold"] = sorted(chunks[i]["id"] for i in g)
        out["rank_before"] = rank_of(g, candidates)
        out["rank_after"] = rank_of(g, reranked)
    if answer:
        out["answer1"] = ask_gemini(query, [i for i, _ in candidates[:TOP_N]])
        out["answer2"] = ask_gemini(query, [i for i, _ in reranked[:TOP_N]])
    return out


def show(res):
    print("QUERY:", res["query"])
    print("gold chunks:", res.get("gold"), "| rank before:", res.get("rank_before"), "-> after:", res.get("rank_after"))
    print("baseline  :", [(chunks[i]["id"], round(s, 3)) for i, s in res["baseline"]])
    print("re-ranked :", [(chunks[i]["id"], round(s, 3)) for i, s in res["reranked"]])
    if "answer1" in res:
        print("\nANSWER 1 (baseline):\n", res["answer1"])
        print("\nANSWER 2 (re-ranked):\n", res["answer2"])

## 5. Failure-mode queries

Each entry: category, query, and `gold_phrase`, a short exact phrase from the rulebook that sits in the chunk holding the true answer. Fill this list after reading the rulebook; keep at least two queries where the gold chunk is at baseline rank 4-10 and ends in the top 3.

Categories: `keyword_trap`, `exception`, `negation`, `buried`.

In [8]:
QUERIES = [
    {"category": "keyword_trap", "query": "Which status makes a figure lose one life at the start of its turn?", "gold_phrase": "suffers 1 damage at the start"},
    {"category": "buried", "query": "Which positive effects can a poisoned monster still receive?", "gold_phrase": "poison prevents the heal"},
    {"category": "exception", "query": "Must the party always go to Gloomhaven after a scenario?", "gold_phrase": "except in the following situations"},
    {"category": "exception", "query": "What happens to retaliate if the retaliating figure is killed?", "gold_phrase": "If a retaliating figure is killed or exhausted"},
    {"category": "negation", "query": "Can loot left on the map be collected after the scenario ends?", "gold_phrase": "they cannot be looted"},
    {"category": "negation", "query": "Can the party lose more reputation than they have?", "gold_phrase": "they cannot lose more than they have"},
    {"category": "control (re-rank hurts)", "query": "Can a figure stand inside a wall line?", "gold_phrase": "figures cannot cross the wall lines"},
    {"category": "control (re-rank hurts)", "query": "Is a road event drawn when the scenario is in casual mode?", "gold_phrase": "any scenario in casual mode"},
]
assert QUERIES, "Add failure queries first (see the cell above)."

In [9]:
results = []
for q in QUERIES:
    res = run_query(q["query"], q["gold_phrase"])
    res["category"] = q["category"]
    results.append(res)
    show(res)
    print("=" * 100)
    time.sleep(5)  # stay under free-tier rate limits

QUERY: Which status makes a figure lose one life at the start of its turn?
gold chunks: ['C062', 'C063'] | rank before: 6 -> after: 1
baseline  : [('C060', 0.432), ('C092', 0.417), ('C142', 0.414), ('C091', 0.409), ('C139', 0.381), ('C063', 0.37), ('C152', 0.359), ('C144', 0.354), ('C093', 0.344), ('C151', 0.337)]
re-ranked : [('C063', 0.727), ('C060', -3.363), ('C092', -4.25), ('C091', -6.158), ('C093', -6.598), ('C152', -7.029), ('C142', -10.731), ('C144', -10.963), ('C151', -10.981), ('C139', -11.143)]

ANSWER 1 (baseline):
 Based on the provided context, there is no mention of a status that makes a figure lose one life at the start of its turn.

ANSWER 2 (re-ranked):
 Based on the provided context, the Wound condition causes a figure to suffer 1 damage (lose 1 life/hit point value) at the start of each of their turns [C063].


QUERY: Which positive effects can a poisoned monster still receive?
gold chunks: ['C063'] | rank before: 6 -> after: 2
baseline  : [('C111', 0.422), ('C093', 0.412), ('C104', 0.399), ('C065', 0.394), ('C161', 0.347), ('C063', 0.344), ('C095', 0.339), ('C110', 0.337), ('C060', 0.319), ('C099', 0.313)]
re-ranked : [('C060', -2.061), ('C063', -3.157), ('C111', -3.484), ('C161', -4.785), ('C065', -4.852), ('C104', -6.3), ('C095', -6.472), ('C093', -7.258), ('C099', -7.794), ('C110', -10.657)]

ANSWER 1 (baseline):
 The context does not contain the answer.

ANSWER 2 (re-ranked):
 Based on the provided context, the text does not state which positive effects a poisoned monster can still receive. It only mentions that poison is removed when the figure is healed, but prevents the heal from increasing the figure's current hit point value [C063].


QUERY: Must the party always go to Gloomhaven after a scenario?
gold chunks: ['C133'] | rank before: 5 -> after: 2
baseline  : [('C119', 0.612), ('C117', 0.561), ('C009', 0.552), ('C002', 0.547), ('C133', 0.522), ('C121', 0.497), ('C147', 0.496), ('C123', 0.469), ('C021', 0.466), ('C010', 0.465)]
re-ranked : [('C119', 7.349), ('C133', 6.042), ('C147', 5.246), ('C117', 5.19), ('C009', 3.554), ('C021', 2.2), ('C010', 1.943), ('C002', 1.0), ('C121', -0.037), ('C123', -0.072)]

ANSWER 1 (baseline):
 Based on the provided context, the party does not always go to Gloomhaven after a scenario. When replaying a scenario ("Replay the Scenario"), the City Phase does not occur, and the party must attempt the scenario again rather than returning to Gloomhaven [C117]. Additionally, if another scenario is force-linked to the scenario, the party plays it instead [C119]. However, normally the party returns to Gloomhaven for a City Phase after completing a scenario [C119], and a City Phase also occurs i

QUERY: What happens to retaliate if the retaliating figure is killed?
gold chunks: ['C069'] | rank before: 5 -> after: 1
baseline  : [('C111', 0.419), ('C070', 0.414), ('C076', 0.407), ('C062', 0.405), ('C069', 0.378), ('C026', 0.377), ('C060', 0.372), ('C063', 0.359), ('C116', 0.35), ('C077', 0.341)]
re-ranked : [('C069', 3.549), ('C070', 1.634), ('C111', 0.277), ('C063', -3.241), ('C062', -4.91), ('C060', -6.443), ('C077', -8.715), ('C076', -8.859), ('C026', -9.213), ('C116', -9.597)]

ANSWER 1 (baseline):
 The provided context does not contain the answer to what happens to retaliate if the retaliating figure is killed.

ANSWER 2 (re-ranked):
 If the retaliating figure is killed by an attack, the retaliate bonus does not occur [C069].


QUERY: Can loot left on the map be collected after the scenario ends?
gold chunks: ['C117'] | rank before: 5 -> after: 1
baseline  : [('C120', 0.483), ('C119', 0.48), ('C116', 0.446), ('C030', 0.434), ('C117', 0.422), ('C133', 0.415), ('C091', 0.387), ('C075', 0.386), ('C084', 0.384), ('C074', 0.381)]
re-ranked : [('C117', 0.667), ('C119', 0.481), ('C116', -1.063), ('C074', -1.082), ('C030', -1.655), ('C084', -3.055), ('C120', -3.101), ('C075', -6.758), ('C133', -7.247), ('C091', -10.144)]

ANSWER 1 (baseline):
 Based on the provided context, there is no mention of whether loot left on the map can be collected after the scenario ends. Therefore, the context does not contain the answer.

ANSWER 2 (re-ranked):
 Based on the provided context, if there are any money tokens or treasure tiles still on the map when the scenario ends, they cannot be looted, as looting can only be performed during the scenario [C117].


QUERY: Can the party lose more reputation than they have?
gold chunks: ['C119'] | rank before: 7 -> after: 4
baseline  : [('C127', 0.42), ('C128', 0.324), ('C082', 0.29), ('C139', 0.285), ('C130', 0.253), ('C151', 0.236), ('C119', 0.227), ('C146', 0.227), ('C144', 0.209), ('C173', 0.202)]
re-ranked : [('C128', 5.282), ('C082', 2.93), ('C127', -1.009), ('C119', -2.803), ('C130', -4.683), ('C146', -7.734), ('C139', -8.467), ('C151', -9.484), ('C144', -10.789), ('C173', -10.897)]

ANSWER 1 (baseline):
 Based on the provided context, the party cannot have less than -10 reputation with any faction, and further decreases beyond -10 are ignored [C127, C128].

ANSWER 2 (re-ranked):
 Based on the provided context, you cannot have less than -10 reputation with any faction, and further decreases are ignored [C128, C127]. The context does not explicitly state whether the party can lose more reputation than they currently have beyond these limits.


QUERY: Can a figure stand inside a wall line?
gold chunks: ['C029'] | rank before: 1 -> after: 4
baseline  : [('C029', 0.408), ('C040', 0.358), ('C042', 0.333), ('C024', 0.333), ('C050', 0.309), ('C049', 0.308), ('C025', 0.303), ('C041', 0.282), ('C079', 0.28), ('C043', 0.277)]
re-ranked : [('C040', -0.323), ('C049', -1.268), ('C042', -2.165), ('C029', -2.879), ('C050', -3.39), ('C041', -4.339), ('C024', -4.975), ('C079', -5.464), ('C025', -7.648), ('C043', -9.583)]

ANSWER 1 (baseline):
 Based on the provided context, figures cannot cross wall lines into a wall hex by any means, even when flying or jumping [C029].

ANSWER 2 (re-ranked):
 Based on the provided context, there is no mention of whether a figure can stand inside a wall line. Therefore, the context does not contain the answer.


QUERY: Is a road event drawn when the scenario is in casual mode?
gold chunks: ['C021'] | rank before: 2 -> after: 4
baseline  : [('C022', 0.48), ('C021', 0.471), ('C147', 0.458), ('C150', 0.408), ('C009', 0.397), ('C014', 0.367), ('C133', 0.33), ('C116', 0.321), ('C123', 0.313), ('C122', 0.304)]
re-ranked : [('C022', 5.687), ('C147', 2.893), ('C150', 2.828), ('C021', 1.113), ('C009', 0.147), ('C014', -2.484), ('C116', -4.943), ('C133', -6.184), ('C123', -7.309), ('C122', -7.732)]

ANSWER 1 (baseline):
 No, no event is resolved when playing any scenario in casual mode [C021, C147].

ANSWER 2 (re-ranked):
 Based on the provided context, no road event is resolved when playing in casual mode [C147, C150].


### Gold chunk text, for judging Answer 1 vs Answer 2 by hand

In [10]:
for r in results:
    print(r["query"])
    for cid in r["gold"]:
        c = chunks[int(cid[1:])]
        print(f"  [{cid}] p.{c['page']}: {c['text'][:600]}")
    print()

Which status makes a figure lose one life at the start of its turn?
  [C062] p.26: amount of damage (rounded down), and ward is then removed. Invisible: The figure cannot be focused on or targeted by any enemy, though non-targeted abilities remain unaffected. The figure and their allies can still interact with each other.Invisible is removed at the end of the figure's next turn. Enemies treat figures with invisible as if they were not there; they can move through figures with invisible but still cannot end their movement in the same hex. Strengthen: The figure gains advantage on all of their attacks. Strengthen is removed at the end of thefigure's next turn. Bless: The figur
  [C063] p.26: there are no bless cards available,bless has no effect. Bless Card NEGATIUE CONDITIONS Wound: The figure suffers 1 damage at the start of each of their turns. Wound is removed when the figure is healed. Poison: All attacks targeting the figure gain “+1 " Poison is removed when the figure is healed bu

## 6. Latency profiling

Warm-up run first (model load / first-call overhead), then the mean over all queries and 5 repeats. LLM time is measured separately because it is the same for both pipelines.

In [11]:
REPEATS = 5
baseline_t, rerank_t = [], []

baseline_retrieve(QUERIES[0]["query"])  # warm-up
rerank(QUERIES[0]["query"], baseline_retrieve(QUERIES[0]["query"]))

for q in QUERIES:
    for _ in range(REPEATS):
        t0 = time.perf_counter()
        cand = baseline_retrieve(q["query"])
        t1 = time.perf_counter()
        rerank(q["query"], cand)
        t2 = time.perf_counter()
        baseline_t.append(t1 - t0)
        rerank_t.append(t2 - t1)

latency = pd.DataFrame(
    {
        "Stage": ["Baseline retrieval (embed query + top-10)", "Cross-encoder re-rank (10 pairs)", "Pipeline 2 total"],
        "Mean (ms)": [np.mean(baseline_t) * 1000, np.mean(rerank_t) * 1000, (np.mean(baseline_t) + np.mean(rerank_t)) * 1000],
        "Std (ms)": [np.std(baseline_t) * 1000, np.std(rerank_t) * 1000, np.nan],
    }
).round(1)
latency

,Stage,Mean (ms),Std (ms)
0,Baseline retrieval (embed query + top-10),16.2,2.9
1,Cross-encoder re-rank (10 pairs),568.3,143.9
2,Pipeline 2 total,584.5,NaN


## 7. Export tables for the LaTeX report

In [12]:
rank_table = pd.DataFrame(
    [
        {
            "Category": r["category"],
            "Query": r["query"],
            "Gold chunk": ", ".join(r["gold"]),
            "Rank before": r["rank_before"],
            "Rank after": r["rank_after"],
        }
        for r in results
    ]
)
display(rank_table)

os.makedirs("report", exist_ok=True)
rank_table.to_latex("report/rank_table.tex", index=False, escape=True, column_format="lp{6cm}lcc")
latency.to_latex("report/latency_table.tex", index=False, escape=True, na_rep="--", float_format="%.1f")

,Category,Query,Gold chunk,Rank before,Rank after
0,keyword_trap,Which status makes a figure lose one life at t...,"C062, C063",6,1
1,buried,Which positive effects can a poisoned monster ...,C063,6,2
2,exception,Must the party always go to Gloomhaven after a...,C133,5,2
3,exception,What happens to retaliate if the retaliating f...,C069,5,1
4,negation,Can loot left on the map be collected after th...,C117,5,1
5,negation,Can the party lose more reputation than they h...,C119,7,4
6,control (re-rank hurts),Can a figure stand inside a wall line?,C029,1,4
7,control (re-rank hurts),Is a road event drawn when the scenario is in ...,C021,2,4
